# QDMpy ODMR Processing Tutorial

This tutorial demonstrates how to use QDMpy's ODMR processing framework to analyze Quantum Diamond Microscopy data. We'll focus on showing how to:

1. Load ODMR data
2. Apply different processors in a pipeline
3. Visualize the results at each step
4. Configure and optimize the fluorescence correction

QDMpy provides a modular, extensible framework for ODMR data processing that lets you build customized workflows for your specific needs.

## Setup and Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# For nicer plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Import QDMpy components
from QDMpy.odmr.data import ODMRData
from QDMpy.odmr.io import MatlabLoader
from QDMpy.odmr.odmr import ODMR
from QDMpy.odmr.processors import (
    BinningProcessor,
    FluorescenceCorrectionProcessor,
    NormalizationProcessor,
    ODMRProcessorManager,
    OutlierProcessor,
    analyze_fluorescence_effects,
    preview_fluorescence_correction
)

## Creating Synthetic ODMR Data

Let's create some synthetic ODMR data to demonstrate the processing capabilities. In real applications, you would load your data from files instead.

In [ ]:
# Parameters for synthetic data
n_polarities = 2          # Number of polarization states
n_freq_ranges = 1         # Number of frequency ranges
rows, cols = 32, 32       # Image dimensions
n_pixels = rows * cols    # Total number of pixels
n_frequencies = 200       # Number of frequency points per sweep

# Create frequency axis (2.87-2.89 GHz range)
frequencies = np.linspace(2.87e9, 2.89e9, n_frequencies)
scan_dimensions = np.array([rows, cols], dtype=int)

# Create base synthetic data
raw_data = np.ones((n_polarities, n_freq_ranges, n_pixels, n_frequencies))

# Create two NV resonance dips in the spectra
freq_centers = [2.87e9 + 0.01e9, 2.87e9 + 0.015e9]
for i in range(n_pixels):
    # Add dips at two positions with varying amplitude and width
    for fc in freq_centers:
        dip_amplitude = 0.06 + 0.03 * np.random.randn()
        dip_width = 4e6 * (1 + 0.2 * np.random.randn())
        raw_data[0, 0, i, :] -= dip_amplitude * np.exp(-((frequencies - fc) ** 2) / (2 * dip_width ** 2))
        # Second polarity has slightly shifted resonances
        raw_data[1, 0, i, :] -= dip_amplitude * np.exp(-((frequencies - (fc + 1e7)) ** 2) / (2 * dip_width ** 2))
    
    # Add random noise
    raw_data[0, 0, i, :] += 0.008 * np.random.randn(n_frequencies)
    raw_data[1, 0, i, :] += 0.008 * np.random.randn(n_frequencies)

# Create a spatial pattern of global fluorescence variation
x = np.linspace(-1, 1, rows)
y = np.linspace(-1, 1, cols)
xx, yy = np.meshgrid(x, y)
# Gaussian pattern centered in the image
spatial_pattern = np.exp(-(xx**2 + yy**2)/0.5).flatten()

# Apply spatial pattern to all frequencies
for p in range(n_polarities):
    for f in range(n_freq_ranges):
        for i in range(n_frequencies):
            raw_data[p, f, :, i] *= 0.7 + 0.5 * spatial_pattern

# Add some outliers
n_outliers = 50
outlier_pixels = np.random.randint(0, n_pixels, n_outliers)
outlier_freqs = np.random.randint(0, n_frequencies, n_outliers)
for p in range(n_polarities):
    for i in range(n_outliers):
        raw_data[p, 0, outlier_pixels[i], outlier_freqs[i]] += 0.5 * np.random.rand()

# Create an ODMRData object
odmr_data = ODMRData(raw_data, scan_dimensions, frequencies)

print(f"Created synthetic ODMR data with shape: {odmr_data.data.shape}")
print(f"Scan dimensions: {odmr_data.scan_dimensions}")
print(f"Frequency range: {odmr_data.frequencies.min()/1e9:.4f} - {odmr_data.frequencies.max()/1e9:.4f} GHz")

## Visualization Functions

Let's define some helper functions to visualize the data at different stages of processing.

In [ ]:
def plot_odmr_spectra(data, frequencies, pixels=None, title="ODMR Spectra"):
    """Plot ODMR spectra for specific pixels."""
    # If no pixels specified, choose random ones
    n_pixels = data.shape[2]
    if pixels is None:
        pixels = np.random.choice(n_pixels, size=4, replace=False)
    
    # Create subplots for each polarity and frequency range
    fig, axes = plt.subplots(data.shape[0], data.shape[1], figsize=(12, 5*data.shape[0]))
    if data.shape[0] == 1 and data.shape[1] == 1:
        axes = np.array([[axes]])  # Ensure 2D array for indexing
    elif data.shape[0] == 1:
        axes = np.array([axes])
    elif data.shape[1] == 1:
        axes = np.array([axes]).T
    
    # Plot individual pixels and the mean
    for p in range(data.shape[0]):  # Polarities
        for f in range(data.shape[1]):  # Frequency ranges
            # Plot selected pixels
            for pixel in pixels:
                axes[p, f].plot(frequencies/1e9, data[p, f, pixel, :], 
                               alpha=0.6, label=f"Pixel {pixel}")
            
            # Plot mean spectrum
            mean_spectrum = np.nanmean(data[p, f], axis=0)
            axes[p, f].plot(frequencies/1e9, mean_spectrum, 'k-', linewidth=2, label="Mean")
            
            # Add labels and legend
            polarity_label = {0: "+", 1: "-"}.get(p, f"P{p}")
            frange_label = {0: "Low", 1: "High"}.get(f, f"F{f}")
            axes[p, f].set_title(f"Polarity: {polarity_label}, Frequency Range: {frange_label}")
            axes[p, f].set_xlabel("Frequency (GHz)")
            axes[p, f].set_ylabel("ODMR Signal")
            axes[p, f].legend()
            axes[p, f].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

def plot_odmr_image(data, scan_dimensions, freq_idx=None, title="ODMR Image"):
    """Plot ODMR data as spatial images."""
    # Use middle frequency by default
    if freq_idx is None:
        freq_idx = data.shape[3] // 2
    
    # Create subplots for each polarity and frequency range
    fig, axes = plt.subplots(data.shape[0], data.shape[1], figsize=(5*data.shape[1], 4*data.shape[0]))
    if data.shape[0] == 1 and data.shape[1] == 1:
        axes = np.array([[axes]])  # Ensure 2D array for indexing
    elif data.shape[0] == 1:
        axes = np.array([axes])
    elif data.shape[1] == 1:
        axes = np.array([axes]).T
    
    # Plot images
    for p in range(data.shape[0]):  # Polarities
        for f in range(data.shape[1]):  # Frequency ranges
            # Reshape the flat pixels to 2D image
            try:
                image = data[p, f, :, freq_idx].reshape(scan_dimensions)
                im = axes[p, f].imshow(image, cmap='viridis')
                plt.colorbar(im, ax=axes[p, f])
                
                # Add labels
                polarity_label = {0: "+", 1: "-"}.get(p, f"P{p}")
                frange_label = {0: "Low", 1: "High"}.get(f, f"F{f}")
                axes[p, f].set_title(f"Polarity: {polarity_label}, Range: {frange_label}, Freq idx: {freq_idx}")
            except ValueError as e:
                # Handle errors (e.g., when dimensions change due to binning)
                print(f"Error reshaping: {e}")
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# Plot the raw data
plot_odmr_spectra(odmr_data.data, odmr_data.frequencies, title="Raw ODMR Spectra")
plot_odmr_image(odmr_data.data, odmr_data.scan_dimensions, title="Raw ODMR Image")

## Setting Up the Processing Pipeline

Now let's create an ODMR processing pipeline using the ODMRProcessorManager.

In [ ]:
# Create an ODMR instance with our data
odmr = ODMR(odmr_data)

# Initialize an empty processor manager
odmr.processor_manager = ODMRProcessorManager()

print("Initialized ODMR instance with empty processor pipeline.")

## 1. Outlier Rejection

First, let's remove outliers from the data. The OutlierProcessor identifies and masks extreme values that might negatively affect downstream processing.

In [ ]:
# Reset and add the outlier processor
odmr.reset()
odmr.processor_manager = ODMRProcessorManager()
odmr.processor_manager.add_processor(OutlierProcessor(threshold=0.015))

# Process the data
odmr.process_data()

# Visualize results
print("Applied outlier rejection with threshold=0.015")
plot_odmr_spectra(odmr.processed_data.data, odmr.processed_data.frequencies, 
                 title="ODMR Spectra After Outlier Rejection")
plot_odmr_image(odmr.processed_data.data, odmr.processed_data.scan_dimensions, 
               title="ODMR Image After Outlier Rejection")

## 2. Fluorescence Correction

Next, let's analyze the fluorescence patterns in our data and apply correction. First, we'll use the analysis functions to determine the optimal correction parameters.

In [ ]:
# Reset to raw data
odmr.reset()

# Analyze fluorescence effects
try:
    pixel_idx, baseline_corrected = analyze_fluorescence_effects(odmr.raw_data)
    print(f"Analysis selected pixel {pixel_idx} as representative")
    print(f"Baseline-corrected data shape: {baseline_corrected.shape}")
    
    # Preview different correction factors
    for factor in [0.0, 0.3, 0.6, 1.0]:
        print(f"\nPreviewing fluorescence correction with factor: {factor}")
        preview_fluorescence_correction(odmr.raw_data, correction_factor=factor, pixel_idx=pixel_idx)
except Exception as e:
    print(f"Error in fluorescence analysis: {e}")
    pixel_idx = odmr.raw_data.data.shape[2] // 2
    print(f"Falling back to middle pixel: {pixel_idx}")

In [ ]:
# Apply the fluorescence correction with the optimal factor
odmr.processor_manager = ODMRProcessorManager()
odmr.processor_manager.add_processor(OutlierProcessor(threshold=0.015))  # Apply outlier rejection first
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor(correction_factor=0.6))

# Process the data
odmr.process_data()

# Visualize results
print("Applied fluorescence correction with factor=0.6")
plot_odmr_spectra(odmr.processed_data.data, odmr.processed_data.frequencies, 
                 title="ODMR Spectra After Fluorescence Correction")
plot_odmr_image(odmr.processed_data.data, odmr.processed_data.scan_dimensions, 
               title="ODMR Image After Fluorescence Correction")

## 3. Normalization

Now let's normalize the spectra to have a consistent scale across pixels.

In [ ]:
# Reset and create a pipeline with outlier rejection, fluorescence correction, and normalization
odmr.reset()
odmr.processor_manager = ODMRProcessorManager()
odmr.processor_manager.add_processor(OutlierProcessor(threshold=0.015))
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor(correction_factor=0.6))
odmr.processor_manager.add_processor(NormalizationProcessor(method='max'))

# Process the data
odmr.process_data()

# Visualize results
print("Applied outlier rejection, fluorescence correction, and normalization")
plot_odmr_spectra(odmr.processed_data.data, odmr.processed_data.frequencies, 
                 title="ODMR Spectra After Normalization")
plot_odmr_image(odmr.processed_data.data, odmr.processed_data.scan_dimensions, 
               title="ODMR Image After Normalization")

## 4. Spatial Binning

Finally, let's apply spatial binning to improve the signal-to-noise ratio. This combines neighboring pixels, reducing spatial resolution but increasing signal quality.

In [ ]:
# Reset and build complete pipeline
odmr.reset()
odmr.processor_manager = ODMRProcessorManager()
odmr.processor_manager.add_processor(OutlierProcessor(threshold=0.015))
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor(correction_factor=0.6))
odmr.processor_manager.add_processor(NormalizationProcessor(method='max'))
odmr.processor_manager.add_processor(BinningProcessor(bin_factor=2))

# List the processors in our pipeline
print("Complete processing pipeline:")
for i, processor in enumerate(odmr.processor_manager.list_processors()):
    print(f"  {i+1}. {processor}")

# Process the data
odmr.process_data()

# Visualize results
print(f"\nOriginal data shape: {odmr.raw_data.data.shape}")
print(f"Processed data shape: {odmr.processed_data.data.shape}")

plot_odmr_spectra(odmr.processed_data.data, odmr.processed_data.frequencies, 
                 title="ODMR Spectra After Complete Processing")

# Create adjusted scan dimensions for the binned data
binned_dims = (odmr.raw_data.scan_dimensions[0] // 2, odmr.raw_data.scan_dimensions[1] // 2)
plot_odmr_image(odmr.processed_data.data, binned_dims, 
               title="ODMR Image After Complete Processing")

## Contrast Enhancement Analysis

Let's look at how our processing pipeline affects the contrast of the ODMR signals, which is a key metric for extracting useful information.

In [ ]:
# Function to calculate and display contrast maps
def plot_contrast_maps(data, scan_dims, title="ODMR Contrast Maps"):
    # Calculate contrast (max - min) for each pixel
    contrast_map = np.zeros((data.shape[0], data.shape[1], data.shape[2]))
    for p in range(data.shape[0]):  # Polarities
        for f in range(data.shape[1]):  # Frequency ranges
            max_values = np.nanmax(data[p, f], axis=1)
            min_values = np.nanmin(data[p, f], axis=1)
            contrast_map[p, f] = max_values - min_values
    
    # Create subplots
    fig, axes = plt.subplots(contrast_map.shape[0], contrast_map.shape[1], 
                            figsize=(5*contrast_map.shape[1], 4*contrast_map.shape[0]))
    if contrast_map.shape[0] == 1 and contrast_map.shape[1] == 1:
        axes = np.array([[axes]])  # Ensure 2D array for indexing
    elif contrast_map.shape[0] == 1:
        axes = np.array([axes])
    elif contrast_map.shape[1] == 1:
        axes = np.array([axes]).T
    
    # Plot each contrast map
    for p in range(contrast_map.shape[0]):  # Polarities
        for f in range(contrast_map.shape[1]):  # Frequency ranges
            try:
                img = contrast_map[p, f].reshape(scan_dims)
                im = axes[p, f].imshow(img, cmap='hot')
                plt.colorbar(im, ax=axes[p, f])
                
                # Add labels
                polarity_label = {0: "+", 1: "-"}.get(p, f"P{p}")
                frange_label = {0: "Low", 1: "High"}.get(f, f"F{f}")
                axes[p, f].set_title(f"Contrast Map (Pol: {polarity_label}, Range: {frange_label})")
                
                # Add mean contrast value
                mean_contrast = np.nanmean(contrast_map[p, f])
                axes[p, f].text(0.05, 0.95, f"Mean contrast: {mean_contrast:.4f}", 
                              transform=axes[p, f].transAxes, 
                              bbox=dict(facecolor='white', alpha=0.7))
            except ValueError as e:
                print(f"Error reshaping: {e}")
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# Compare contrast between raw and processed data
print("Raw Data Contrast:")
plot_contrast_maps(odmr.raw_data.data, odmr.raw_data.scan_dimensions, title="Raw Data Contrast Maps")

print("Processed Data Contrast:")
plot_contrast_maps(odmr.processed_data.data, binned_dims, title="Processed Data Contrast Maps")

## Metadata Tracking

One advantage of the QDMpy processor framework is automatic tracking of processing history in the metadata.

In [ ]:
print("Processed data metadata:")
for key, value in odmr.processed_data.metadata.items():
    print(f"\n{key}:")
    if isinstance(value, dict):
        for subkey, subvalue in value.items():
            print(f"  {subkey}: {subvalue}")
    else:
        print(f"  {value}")

## Conclusion

In this tutorial, we demonstrated QDMpy's modular ODMR data processing framework. The key steps for effective ODMR data processing are:

1. **Outlier Rejection**: Remove anomalous data points that could skew results
2. **Fluorescence Correction**: Compensate for global fluorescence variations using the `analyze_fluorescence_effects` and `preview_fluorescence_correction` functions
3. **Normalization**: Standardize signal levels across pixels for consistent analysis
4. **Spatial Binning**: Combine neighboring pixels to improve signal quality at the cost of spatial resolution

The QDMpy processor framework provides a flexible, extensible way to build custom data processing pipelines. The modular design makes it easy to add, remove, or reorder processing steps as needed for your specific application.